In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import os
from torch.cuda.amp import GradScaler, autocast 

In [7]:
# Hyperparameters
nz = 100
ngf = 64
ndf = 64
num_epochs = 5
lr = 0.0001
beta1 = 0.5
n_critic = 5
clip_value = 0.01
batch_size = 64
num_classes = 5
image_size = 64
num_channels = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
# Label mapping
label_map = {'food': 0, 'drink': 1, 'inside': 2, 'outside': 3, 'menu': 4}
reverse_label_map = {v: k for k, v in label_map.items()}

In [11]:
# Custom Dataset
class CustomImageDataset(Dataset):
    def __init__(self, metadata_file, root_dir, split, transform=None):
        self.metadata = pd.read_csv(metadata_file)
        self.root_dir = os.path.join(root_dir, split)
        self.transform = transform
        self.label_map = label_map

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        img_filename = os.path.basename(self.metadata.iloc[idx]['image_path'])
        label_str = self.metadata.iloc[idx]['label']
        
        if label_str not in self.label_map:
            print(f"Warning: Invalid label '{label_str}' at index {idx}. Skipping.")
            return None
        label = self.label_map[label_str]
        
        img_path = os.path.join(self.root_dir, label_str, img_filename)
        
        try:
            image = Image.open(img_path).convert('L')
        except FileNotFoundError:
            print(f"Warning: Image not found at {img_path}. Skipping.")
            return None
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [13]:
# Data transforms with augmentation
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [15]:
# File paths
root_dir = r"C:\Users\owner\Downloads\processed"  
train_metadata_file = r"C:\Users\owner\Downloads\train_meta.csv"
test_metadata_file = r"C:\Users\owner\Downloads\test_meta.csv" 

In [17]:
# Create datasets and dataloaders with optimization
train_dataset = CustomImageDataset(metadata_file=train_metadata_file, root_dir=root_dir, split='train', transform=transform)
test_dataset = CustomImageDataset(metadata_file=test_metadata_file, root_dir=root_dir, split='test', transform=transform)
train_dataset = [x for x in train_dataset if x is not None]
test_dataset = [x for x in test_dataset if x is not None]
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

In [19]:
# Optimized Generator
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, 50)
        self.init_size = 8
        self.l1 = nn.Sequential(nn.Linear(nz + 50, 64 * self.init_size * self.init_size))
        
        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(32, num_channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, z, labels):
        label_input = self.label_emb(labels)
        gen_input = torch.cat((z, label_input), dim=1)
        out = self.l1(gen_input)
        out = out.view(out.shape[0], 64, self.init_size, self.init_size)
        img = self.conv_blocks(out)
        img = self.upsample(img)
        return img

In [21]:
# Optimized Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, 50)
        self.conv_blocks = nn.Sequential(
            nn.Conv2d(num_channels, 32, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 16 * 16 + 50, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1)
        )

    def forward(self, img, labels):
        out = self.conv_blocks(img)
        out = out.view(out.shape[0], -1)
        label_input = self.label_emb(labels)
        d_in = torch.cat((out, label_input), dim=1)
        validity = self.fc(d_in)
        return validity

In [23]:
# Initialize models and optimizers with scheduler
generator = Generator().to(device)
discriminator = Discriminator().to(device)
optimizer_G = optim.RMSprop(generator.parameters(), lr=lr)
optimizer_D = optim.RMSprop(discriminator.parameters(), lr=lr)
scheduler_G = optim.lr_scheduler.ReduceLROnPlateau(optimizer_G, 'min', factor=0.5, patience=5)
scheduler_D = optim.lr_scheduler.ReduceLROnPlateau(optimizer_D, 'min', factor=0.5, patience=5)

In [25]:
# Training loop with conditional mixed precision
scaler = GradScaler() if torch.cuda.is_available() else None

for epoch in range(num_epochs):
    for i, batch in enumerate(train_dataloader):
        if batch is None:
            continue
        real_imgs, labels = batch
        batch_size = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        labels = labels.to(device)

        # Train Discriminator
        for _ in range(n_critic):
            optimizer_D.zero_grad()
            if torch.cuda.is_available() and scaler is not None:
                with autocast():
                    real_validity = discriminator(real_imgs, labels)
                    d_real_loss = -torch.mean(real_validity)
                    
                    z = torch.randn(batch_size, nz).to(device)
                    fake_labels = torch.randint(0, num_classes, (batch_size,)).to(device)
                    fake_imgs = generator(z, fake_labels)
                    fake_validity = discriminator(fake_imgs.detach(), fake_labels)
                    d_fake_loss = torch.mean(fake_validity)
                    
                    d_loss = d_real_loss + d_fake_loss
                scaler.scale(d_loss).backward()
                scaler.step(optimizer_D)
                scaler.update()
            else:
                real_validity = discriminator(real_imgs, labels)
                d_real_loss = -torch.mean(real_validity)
                
                z = torch.randn(batch_size, nz).to(device)
                fake_labels = torch.randint(0, num_classes, (batch_size,)).to(device)
                fake_imgs = generator(z, fake_labels)
                fake_validity = discriminator(fake_imgs.detach(), fake_labels)
                d_fake_loss = torch.mean(fake_validity)
                
                d_loss = d_real_loss + d_fake_loss
                d_loss.backward()
                optimizer_D.step()
            
            for p in discriminator.parameters():
                p.data.clamp_(-clip_value, clip_value)

        # Train Generator
        optimizer_G.zero_grad()
        if torch.cuda.is_available() and scaler is not None:
            with autocast():
                fake_validity = discriminator(fake_imgs, fake_labels)
                g_loss = -torch.mean(fake_validity)
            scaler.scale(g_loss).backward()
            scaler.step(optimizer_G)
            scaler.update()
        else:
            fake_validity = discriminator(fake_imgs, fake_labels)
            g_loss = -torch.mean(fake_validity)
            g_loss.backward()
            optimizer_G.step()

        scheduler_G.step(g_loss)
        scheduler_D.step(d_loss)

        if i % 100 == 0:
            print(f"[Epoch {epoch}/{num_epochs}] [Batch {i}/{len(train_dataloader)}] "
                  f"D loss: {d_loss.item():.4f}, G loss: {g_loss.item():.4f}")

C:\Users\owner\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[Epoch 0/5] [Batch 0/2500] D loss: -0.0442, G loss: 0.0223
[Epoch 0/5] [Batch 100/2500] D loss: -1.9843, G loss: 1.1021
[Epoch 0/5] [Batch 200/2500] D loss: -2.2111, G loss: 1.2099
[Epoch 0/5] [Batch 300/2500] D loss: -2.1645, G loss: 1.2200
[Epoch 0/5] [Batch 400/2500] D loss: -2.2253, G loss: 1.2056
[Epoch 0/5] [Batch 500/2500] D loss: -2.1832, G loss: 1.2145
[Epoch 0/5] [Batch 600/2500] D loss: -2.1905, G loss: 1.2176
[Epoch 0/5] [Batch 700/2500] D loss: -2.1984, G loss: 1.2083
[Epoch 0/5] [Batch 800/2500] D loss: -2.1996, G loss: 1.2147
[Epoch 0/5] [Batch 900/2500] D loss: -2.1975, G loss: 1.2231
[Epoch 0/5] [Batch 1000/2500] D loss: -2.2111, G loss: 1.2108
[Epoch 0/5] [Batch 1100/2500] D loss: -2.1531, G loss: 1.2022
[Epoch 0/5] [Batch 1200/2500] D loss: -2.1967, G loss: 1.2222
[Epoch 0/5] [Batch 1300/2500] D loss: -2.1755, G loss: 1.2211
[Epoch 0/5] [Batch 1400/2500] D loss: -2.2119, G loss: 1.2113
[Epoch 0/5] [Batch 1500/2500] D loss: -2.1772, G loss: 1.2179
[Epoch 0/5] [Batch 1

In [37]:
# Generate 10 images for each class
for label_idx, class_name in reverse_label_map.items():
    generate_samples(1, label_idx, class_name)

In [33]:
from torchvision.utils import save_image

In [35]:
# Generate samples function
def generate_samples(num_samples, label_idx, class_name):
    z = torch.randn(num_samples, nz).to(device)
    labels = torch.full((num_samples,), label_idx, dtype=torch.long).to(device)
    with torch.no_grad():
        fake_imgs = generator(z, labels)
    save_image(fake_imgs, f"generated_{class_name}.png", normalize=True)